In [4]:
import pandas as pd
import numpy as np
import seaborn as sns

In [5]:
pd.set_option('display.max_columns', None)
churn=pd.read_csv("Clean_TelcoCustomerChurn.csv")

In [6]:
churn.sample(7)

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,Churn
4258,Female,1,Yes,No,43,Yes,Yes,Fiber optic,Yes,No,Yes,Yes,Yes,Yes,One year,Yes,Mailed check,109.55,Yes
81,Female,0,Yes,Yes,11,No,No,DSL,Yes,No,No,No,Yes,Yes,Month-to-month,No,Electronic check,50.55,No
2086,Male,0,Yes,Yes,17,Yes,No,Fiber optic,No,No,No,No,Yes,No,Month-to-month,No,Electronic check,80.85,Yes
5207,Female,1,Yes,No,24,No,No,DSL,No,No,Yes,No,No,Yes,Month-to-month,Yes,Credit card (automatic),40.65,Yes
1748,Female,0,Yes,No,54,Yes,No,No,No,No,No,No,No,No,Two year,No,Credit card (automatic),20.05,No
5853,Male,0,Yes,No,62,Yes,Yes,Fiber optic,Yes,Yes,No,No,No,Yes,One year,No,Electronic check,96.10,No
4950,Female,0,Yes,No,4,Yes,Yes,Fiber optic,No,Yes,No,No,Yes,No,Month-to-month,Yes,Electronic check,91.65,Yes


In [7]:
churn.shape

(7043, 19)

In [8]:
churn["Churn"].value_counts()

Churn
No     5174
Yes    1869
Name: count, dtype: int64

In [10]:
X=churn.drop(columns="Churn")
y=churn.Churn

In [11]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.3,stratify=y,random_state=42)

# Pipeline

In [13]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder,FunctionTransformer,StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

In [14]:
churn.columns

Index(['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure',
       'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity',
       'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV',
       'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod',
       'MonthlyCharges', 'Churn'],
      dtype='object')

In [15]:
num_col=["tenure","MonthlyCharges"]
cat_col=['gender', 'SeniorCitizen', 'Partner', 'Dependents','PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity',
       'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV','StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']

In [16]:
num_pipe=Pipeline(steps=[
    ("Encode",StandardScaler())
])
cat_pipe=Pipeline(steps=[
    ("Encode",OneHotEncoder(drop="first"))
])

In [17]:
preprocessing=ColumnTransformer(transformers=[
    ("prepro_num",num_pipe,num_col),
    ("prepro_cat",cat_pipe,cat_col)
])

In [18]:
full_pipe=Pipeline(steps=[
    ("preprocessing",preprocessing),
    ("Model",LogisticRegression())
    # ("Model1",DecisionTreeClassifier(max_depth=11))
])

In [19]:
full_pipe.fit(X_train,y_train)

Pipeline(steps=[('preprocessing',
                 ColumnTransformer(transformers=[('prepro_num',
                                                  Pipeline(steps=[('Encode',
                                                                   StandardScaler())]),
                                                  ['tenure', 'MonthlyCharges']),
                                                 ('prepro_cat',
                                                  Pipeline(steps=[('Encode',
                                                                   OneHotEncoder(drop='first'))]),
                                                  ['gender', 'SeniorCitizen',
                                                   'Partner', 'Dependents',
                                                   'PhoneService',
                                                   'MultipleLines',
                                                   'InternetService',
                                                   'OnlineSecurity',
                                                   'OnlineBackup',
                                                   'DeviceProtection',
                                                   'TechSupport', 'StreamingTV',
                                                   'StreamingMovies',
                                                   'Contract',
                                                   'PaperlessBilling',
                                                   'PaymentMethod'])])),
                ('Model', LogisticRegression())])

In [45]:
y_pred=full_pipe.predict(X_test)

In [50]:
# y_pred.tolist()

In [52]:
full_pipe.score(X_train,y_train)

0.8030425963488844

In [54]:
full_pipe.score(X_test,y_test)

0.7998106956933271

In [20]:
a=full_pipe.named_steps["Model"]

In [21]:
a.coef_[0]   # LogisticRegression has coef_, not feature_importances_ (that's a tree-model attribute)

array([-0.77821709, -0.04658223,  0.027259  ,  0.14497175, -0.00597722,
       -0.27351951, -0.3122409 ,  0.33903302,  0.86015796, -0.9743184 ,
       -0.3773226 , -0.12665119, -0.00737449, -0.32722859,  0.24738208,
        0.3367479 , -0.71797619, -1.27827375,  0.40532382, -0.04441324,
        0.38002371,  0.12523773])

In [89]:
# from sklearn.tree import plot_tree

# plot_tree(a,feature_names=full_pipe.named_steps['preprocessing'].get_feature_names_out(),filled=True, 
#     rounded=True, );

In [56]:
from sklearn.model_selection import cross_val_score
from sklearn.metrics import confusion_matrix,roc_auc_score,recall_score,precision_score

In [59]:
(cross_val_score(full_pipe,X_train,y_train,cv=5)).mean()

0.8022312373225153

In [60]:

y_pred=full_pipe.predict(X_test)

In [63]:
confusion_matrix(y_test,y_pred)

array([[1387,  165],
       [ 258,  303]], dtype=int64)

In [65]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))


              precision    recall  f1-score   support

          No       0.84      0.89      0.87      1552
         Yes       0.65      0.54      0.59       561

    accuracy                           0.80      2113
   macro avg       0.75      0.72      0.73      2113
weighted avg       0.79      0.80      0.79      2113



In [72]:
roc_auc_score(y_test, full_pipe.predict_proba(X_test)[:,1])   # needs P(Yes), not the predicted labels

0.8432176525718066

In [68]:
import pickle as pkl

In [70]:
pkl.dump({
    "Model":full_pipe,
    "cat_col":cat_col,
    "num_col":num_col
},open("ChurnModel.pkl","wb"))

In [85]:
temp=full_pipe.named_steps["preprocessing"]

In [87]:
temp.get_feature_names_out()

array(['prepro_num__tenure', 'prepro_num__MonthlyCharges',
       'prepro_cat__gender_Male', 'prepro_cat__SeniorCitizen_1',
       'prepro_cat__Partner_Yes', 'prepro_cat__Dependents_Yes',
       'prepro_cat__PhoneService_Yes', 'prepro_cat__MultipleLines_Yes',
       'prepro_cat__InternetService_Fiber optic',
       'prepro_cat__InternetService_No', 'prepro_cat__OnlineSecurity_Yes',
       'prepro_cat__OnlineBackup_Yes', 'prepro_cat__DeviceProtection_Yes',
       'prepro_cat__TechSupport_Yes', 'prepro_cat__StreamingTV_Yes',
       'prepro_cat__StreamingMovies_Yes', 'prepro_cat__Contract_One year',
       'prepro_cat__Contract_Two year',
       'prepro_cat__PaperlessBilling_Yes',
       'prepro_cat__PaymentMethod_Credit card (automatic)',
       'prepro_cat__PaymentMethod_Electronic check',
       'prepro_cat__PaymentMethod_Mailed check'], dtype=object)

In [ ]:
lr.predict_proba